In [1]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. DIRECTORIES & COLOR SETUP
# ==========================================
base_folder = "C:\\Users\\PC\\Documents\\MSc_Enugu_Data\\"
output_folder = os.path.join(base_folder, "Thesis_Exported_Maps")
os.makedirs(output_folder, exist_ok=True) # Creates a new folder for your PNGs

years = [2000, 2005, 2010, 2015, 2020, 2025]

# LULC Setup
lulc_colors = ['#FF0000', '#FFA500', '#00A000', '#8B4513', '#0000FF']
lulc_cmap = ListedColormap(lulc_colors)
lulc_labels = ['Dense Urban', 'Sparse Urban', 'Vegetation', 'Bare Soil', 'Water Body']
lulc_patches = [mpatches.Patch(color=lulc_colors[i], label=lulc_labels[i]) for i in range(5)]

# UHI Setup
uhi_colors = ['#0000FF', '#FFFFFF', '#FF0000']
uhi_cmap = ListedColormap(uhi_colors)
uhi_labels = ['Coldspot (<-1.5)', 'Neutral (-1.5 to 1.5)', 'Hotspot (>1.5)']
uhi_patches = [mpatches.Patch(color=uhi_colors[i], label=uhi_labels[i]) for i in range(3)]

# ==========================================
# 2. MAP GENERATOR LOOP
# ==========================================
print("Generating high-resolution thesis maps...\n")

for year in years:
    file_path = os.path.join(base_folder, f"Enugu_{year}_COMPLETE_ANALYSIS_STACK.tif")
    
    with rasterio.open(file_path) as src:
        # Read only the required bands: 13 (UHI), 14 (LULC), 15 (Valid Mask)
        # Note: Rasterio uses 1-based indexing, so python indices 12, 13, 14 = Rasterio 13, 14, 15
        uhi_raw = src.read(13)
        lulc_raw = src.read(14)
        valid_mask = src.read(15)
        
    # Mask out dead space (NaNs) outside the city boundary
    city_mask = np.isnan(valid_mask)
    
    # Process LULC
    lulc_clean = np.ma.masked_where(city_mask, lulc_raw)
    
    # Process UHI (Apply your strict neural network classification rules)
    uhi_classes = np.zeros_like(uhi_raw)
    uhi_classes[uhi_raw < -1.5] = 0
    uhi_classes[(uhi_raw >= -1.5) & (uhi_raw <= 1.5)] = 1
    uhi_classes[uhi_raw > 1.5] = 2
    uhi_clean = np.ma.masked_where(city_mask, uhi_classes)
    
    # ==========================================
    # 3. PLOT AND EXPORT LULC
    # ==========================================
    fig, ax = plt.subplots(figsize=(10, 10))
    # Background color for NoData areas (outside city)
    ax.set_facecolor('#E0E0E0') 
    
    ax.imshow(lulc_clean, cmap=lulc_cmap, vmin=0, vmax=4)
    ax.set_title(f"Land Use Land Cover (LULC) - Enugu Metropolis ({year})", fontsize=14, pad=15)
    ax.legend(handles=lulc_patches, loc='lower right', title="LULC Classes", facecolor='white', framealpha=1)
    ax.axis('off')
    
    lulc_output_path = os.path.join(output_folder, f"LULC_Map_{year}.png")
    plt.savefig(lulc_output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # ==========================================
    # 4. PLOT AND EXPORT UHI
    # ==========================================
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_facecolor('#E0E0E0') 
    
    ax.imshow(uhi_clean, cmap=uhi_cmap, vmin=0, vmax=2)
    ax.set_title(f"Urban Heat Island (UHI) Distribution - Enugu Metropolis ({year})", fontsize=14, pad=15)
    ax.legend(handles=uhi_patches, loc='lower right', title="Thermal Intensity", facecolor='white', framealpha=1)
    ax.axis('off')
    
    uhi_output_path = os.path.join(output_folder, f"UHI_Map_{year}.png")
    plt.savefig(uhi_output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Successfully exported {year} maps.")

print(f"\nAll thesis maps saved to: {output_folder}")

Generating high-resolution thesis maps...

Successfully exported 2000 maps.
Successfully exported 2005 maps.
Successfully exported 2010 maps.
Successfully exported 2015 maps.
Successfully exported 2020 maps.
Successfully exported 2025 maps.

All thesis maps saved to: C:\Users\PC\Documents\MSc_Enugu_Data\Thesis_Exported_Maps


In [3]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. DIRECTORIES & COLOR SETUP
# ==========================================
base_folder = "C:\\Users\\PC\\Documents\\MSc_Enugu_Data\\"
output_folder = os.path.join(base_folder, "Thesis_Exported_Maps_V3")
os.makedirs(output_folder, exist_ok=True) 

years = [2000, 2005, 2010, 2015, 2020, 2025]

lulc_colors = ['#FF0000', '#FFA500', '#00A000', '#8B4513', '#0000FF']
lulc_cmap = ListedColormap(lulc_colors)
lulc_labels = ['Dense Urban', 'Sparse Urban', 'Vegetation', 'Bare Soil', 'Water Body']
lulc_patches = [mpatches.Patch(color=lulc_colors[i], label=lulc_labels[i]) for i in range(5)]

uhi_colors = ['#0000FF', '#FFFFFF', '#FF0000']
uhi_cmap = ListedColormap(uhi_colors)
uhi_labels = ['Coldspot (<-1.5)', 'Neutral (-1.5 to 1.5)', 'Hotspot (>1.5)']
uhi_patches = [mpatches.Patch(color=uhi_colors[i], label=uhi_labels[i]) for i in range(3)]

# ==========================================
# 2. MAP GENERATOR LOOP WITH COORDINATES
# ==========================================
print("Generating high-resolution thesis maps with corrected spatial masking...\n")

for year in years:
    file_path = os.path.join(base_folder, f"Enugu_{year}_COMPLETE_ANALYSIS_STACK.tif")
    
    with rasterio.open(file_path) as src:
        uhi_raw = src.read(13)
        lulc_raw = src.read(14)
        
        # Extract geographic bounding box for coordinates
        bounds = src.bounds
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
        
    # THE FIX: Use the raw thermal band to define the city boundary. 
    # If there is no temperature recorded, it is outside the city limits.
    city_mask = np.isnan(uhi_raw)
    
    # Apply the mask to LULC
    lulc_clean = np.ma.masked_where(city_mask, lulc_raw)
    
    # Apply strict classification rules to UHI, then mask
    uhi_classes = np.zeros_like(uhi_raw)
    uhi_classes[uhi_raw < -1.5] = 0
    uhi_classes[(uhi_raw >= -1.5) & (uhi_raw <= 1.5)] = 1
    uhi_classes[uhi_raw > 1.5] = 2
    uhi_clean = np.ma.masked_where(city_mask, uhi_classes)
    
    footer_text = "Project: Ocholi Israel MSc (Geoinformatics & Surveying, UNN) | Data Source: Landsat & Sentinel-2 (GEE)"
    
    # ==========================================
    # 3. PLOT AND EXPORT LULC
    # ==========================================
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_facecolor('#E0E0E0') 
    
    ax.imshow(lulc_clean, cmap=lulc_cmap, vmin=0, vmax=4, extent=extent)
    ax.set_title(f"Land Use Land Cover (LULC) - Enugu Metropolis ({year})", fontsize=14, pad=15)
    
    ax.legend(handles=lulc_patches, loc='upper right', title="LULC Classes", facecolor='white', framealpha=1)
    ax.set_xlabel("Easting / Longitude", fontsize=10)
    ax.set_ylabel("Northing / Latitude", fontsize=10)
    ax.tick_params(axis='both', which='major', labelsize=9)
    ax.grid(color='white', linestyle='--', alpha=0.4) 
    
    fig.text(0.5, 0.05, footer_text, ha='center', fontsize=10, 
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='black', boxstyle='round,pad=0.5'))
    
    plt.savefig(os.path.join(output_folder, f"LULC_Map_{year}_Final.png"), dpi=300, bbox_inches='tight')
    plt.close()
    
    # ==========================================
    # 4. PLOT AND EXPORT UHI
    # ==========================================
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_facecolor('#E0E0E0') 
    
    ax.imshow(uhi_clean, cmap=uhi_cmap, vmin=0, vmax=2, extent=extent)
    ax.set_title(f"Urban Heat Island (UHI) Distribution - Enugu Metropolis ({year})", fontsize=14, pad=15)
    
    ax.legend(handles=uhi_patches, loc='upper right', title="Thermal Intensity", facecolor='white', framealpha=1)
    ax.set_xlabel("Easting / Longitude", fontsize=10)
    ax.set_ylabel("Northing / Latitude", fontsize=10)
    ax.tick_params(axis='both', which='major', labelsize=9)
    ax.grid(color='white', linestyle='--', alpha=0.4)
    
    fig.text(0.5, 0.05, footer_text, ha='center', fontsize=10, 
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='black', boxstyle='round,pad=0.5'))
    
    plt.savefig(os.path.join(output_folder, f"UHI_Map_{year}_Final.png"), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Successfully exported {year} layout.")

print(f"\nFinal thesis maps saved to: {output_folder}")

Generating high-resolution thesis maps with corrected spatial masking...

Successfully exported 2000 layout.
Successfully exported 2005 layout.
Successfully exported 2010 layout.
Successfully exported 2015 layout.
Successfully exported 2020 layout.
Successfully exported 2025 layout.

Final thesis maps saved to: C:\Users\PC\Documents\MSc_Enugu_Data\Thesis_Exported_Maps_V3


In [4]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. DIRECTORY SETUP & INDEX CONFIGURATION
# ==========================================
base_folder = "C:\\Users\\PC\\Documents\\MSc_Enugu_Data\\"
output_folder = os.path.join(base_folder, "Thesis_Exported_Grids")
os.makedirs(output_folder, exist_ok=True) 

years = [2000, 2005, 2010, 2015, 2020, 2025]

# Define the exact file suffixes from your folder and assign scientific colormaps
indices = {
    'LST': {'suffix': 'LST_C', 'cmap': 'RdYlBu_r', 'title': 'Land Surface Temperature (°C)'},
    'NDVI': {'suffix': 'NDVI', 'cmap': 'RdYlGn', 'title': 'Normalized Difference Vegetation Index (NDVI)'},
    'NDBI': {'suffix': 'NDBI', 'cmap': 'YlOrBr', 'title': 'Normalized Difference Built-up Index (NDBI)'}
}

footer_text = "Project: Ocholi Israel MSc (Geoinformatics & Surveying, UNN) | Data Source: Landsat & Sentinel-2 (GEE)"

# ==========================================
# 2. GENERATE 2x3 GRIDS FOR EACH INDEX
# ==========================================
print("Compiling 2x3 Spatiotemporal Grids for Thesis Presentation...\n")

for index_name, props in indices.items():
    # Create a 2x3 grid of subplots that share the same X and Y coordinate axes
    fig, axes = plt.subplots(2, 3, figsize=(22, 14), sharex=True, sharey=True)
    fig.suptitle(f"Spatiotemporal Distribution of {props['title']} (2000 - 2025)", 
                 fontsize=22, fontweight='bold', y=0.96)
    
    # Flatten the 2x3 array of axes into a simple list for easy looping
    axes = axes.flatten()
    
    for idx, year in enumerate(years):
        file_path = os.path.join(base_folder, f"Enugu_{year}_{props['suffix']}.tif")
        
        try:
            with rasterio.open(file_path) as src:
                # Read the single band containing the index values
                data = src.read(1)
                
                # Extract coordinates for the axes
                bounds = src.bounds
                extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
                
            # Mask out the background outside the city boundary
            city_mask = np.isnan(data)
            clean_data = np.ma.masked_where(city_mask, data)
            
            ax = axes[idx]
            ax.set_facecolor('#E0E0E0') 
            
            # Render the map
            im = ax.imshow(clean_data, cmap=props['cmap'], extent=extent)
            ax.set_title(f"{year}", fontsize=18, pad=10, fontweight='bold')
            
            # Attach a colorbar to each individual year to accurately show changing max/min values
            cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.ax.tick_params(labelsize=10)
            
            # Format Coordinates and Gridlines
            ax.tick_params(axis='both', which='major', labelsize=10)
            ax.grid(color='white', linestyle='--', alpha=0.4)
            
            # Only add X-axis labels to the bottom row, and Y-axis labels to the left column
            if idx >= 3: 
                ax.set_xlabel("Easting / Longitude", fontsize=12)
            if idx % 3 == 0: 
                ax.set_ylabel("Northing / Latitude", fontsize=12)
                
        except Exception as e:
            print(f"Error loading {year} {index_name}: Ensure the file exists. ({e})")
            
    # Inject custom MSc Project attribution at the very bottom
    fig.text(0.5, 0.04, footer_text, ha='center', fontsize=14, 
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='black', boxstyle='round,pad=0.5'))
    
    # Adjust layout to prevent overlapping text and save
    plt.subplots_adjust(wspace=0.1, hspace=0.15)
    output_path = os.path.join(output_folder, f"{index_name}_Decadal_Grid.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Successfully exported {index_name} 2x3 Grid.")

print(f"\nAll index grids saved to: {output_folder}")

Compiling 2x3 Spatiotemporal Grids for Thesis Presentation...

Successfully exported LST 2x3 Grid.
Error loading 2025 NDVI: Ensure the file exists. (C:\Users\PC\Documents\MSc_Enugu_Data\Enugu_2025_NDVI.tif: No such file or directory)
Successfully exported NDVI 2x3 Grid.
Error loading 2025 NDBI: Ensure the file exists. (C:\Users\PC\Documents\MSc_Enugu_Data\Enugu_2025_NDBI.tif: No such file or directory)
Successfully exported NDBI 2x3 Grid.

All index grids saved to: C:\Users\PC\Documents\MSc_Enugu_Data\Thesis_Exported_Grids
